# 解答例：ホテルレビューの分析によるレビュー改善・集客改善

対象データ：ホテルレビュー 500件（満足度調査、評価は1〜5の5段階）  
対応するワークブック：`模範解答_ホテルレビュー分析.xlsx`

## 課題設定（前提）

### あなたの立場

あなたはホテルのマーケティング担当です。

### 支配人からの依頼

> 「レビューサイトの点数が伸びない。良い設備は入れているはずなのに、評価が上がらない。
>
> アンケートが500件たまっているので、分析してほしい。
> どこを直せばレビューが上がるのか、どういう客を狙えば集客につながるのか。

### 渡されたデータ

| 列 | 内容 |
|---|---|
| 属性4列 | 性別／年代／個人・グループ／宿泊カテゴリ |
| 評価9項目 | Wifi設備／送迎の便利さ／オンライン予約のしやすさ／ホテルの立地／食事の満足度／チェックイン・アウトの接客／その他のサービス／清潔さ／滞在時の快適さ |
| 目的変数 | 満足度（1〜5） |

## 考え方の枠組み：What → Where → Why → How

分析を「集計の寄せ集め」にしないために、4段階で絞り込む。

各段階で**何を棄却したか**が、採用した判断の根拠になる。下段の点線が棄却した道筋である。

![data](img/10/01.png)

### 4段階とこの分析の対応

| 段階 | 問い | この分析での答え | 使った手法 |
|---|---|---|---|
| **What** | 何が問題か | 平均3.24ではなく、低評価22%（110件）が点数を押し下げている | 分布の確認（平均・中央値・分布） |
| **Where** | どこに集中しているか | 個人客（低評価率33.5%）と Wifi設備（平均2.90で最下位） | セグメント別クロス集計 |
| **Why** | なぜそこなのか | Wifiは影響度2位（係数0.364）なのに点数が9位。単相関では4位で見逃す | 重回帰分析＋単相関との比較 |
| **How** | どうするか | Wifiを1点上げると高評価率 +12.7pt。送迎・立地は±0pt | 回帰式による施策シミュレーション |

### なぜこの順番でなければならないか

- **What を飛ばすと**、平均を上げる施策になる。平均は外れ値に引かれるため、全体の底上げという無理な結論に向かう
- **Where を飛ばすと**、全項目を均等に改善する話になる。予算が限られている以上、成立しない
- **Why を飛ばすと**、単相関の高い順に選ぶ。清潔さが最優先になり、投資先を間違える
- **How を飛ばすと**、効果の見積もりがない提案になる。「で、何が良くなるのか」に答えられない

## 第1部　前処理で気づくべき4点

この演習の採点は、集計の正しさよりも**異常値に気づけたか**に重みがある。

### 判断1　ホテルの立地に 11.54 という値が1件ある

評価は1〜5のはずなのに、1件だけ **11.537967914438502** が入っている。

厄介なのは、**整数ではなく小数**であること。`=COUNTIF(範囲,11)` では見つからない。見つけるには次のいずれかが必要。

- `=MAX(範囲)` で最大値を確認する
- 降順ソートして先頭を見る
- `=COUNTIF(範囲,">5")` で範囲外を数える

**対処**：その1件を空欄にする（または行ごと除外）。

### 判断2　送迎とオンライン予約に 0 がある

| 項目 | 0の件数 |
|---|---|
| 送迎の便利さ | 22件（4.4%） |
| オンライン予約のしやすさ | 8件（1.6%） |

1〜5のスケールに0はない。**「該当なし・未回答」**を意味する。送迎を使わなかった客、電話で予約した客である。

これを最低評価の1として扱うと、その項目の平均が実際より低く出る。

| | 0を含めた平均 | 0を除いた平均 |
|---|---|---|
| 送迎の便利さ | 3.10 | **3.24** |
| オンライン予約のしやすさ | 2.92 | **2.97** |

**対処**：空欄にして平均の母数から除く。回帰では該当行を除外する。

### 判断3　年代に 0 が7件ある

0代は存在しない。年代不明として別扱いにする。ただし満足度そのものの分析からは除外しない。

### 判断4　70代5件・80代1件は率で語れない

80代は1件なので高評価率0%。これを「80代の満足度が最低」と報告してはいけない。

## 第2部　分析結果

### 全体像

| 指標 | 値 |
|---|---|
| 回答件数 | 500件 |
| 満足度 平均 | 3.24 |
| 満足度 中央値 | 3.0 |
| 高評価率（4以上） | 42.2% |
| 低評価率（2以下） | 22.0% |
| 異常値を含む回答 | 29件 |

**平均を上げる最短経路は、低評価の110件を減らすこと。** 平均3.24を3.3にする議論より、2割が何に不満なのかを見るほうが実務的である。

### 評価項目の現状（クレンジング後）

| 項目 | 平均 | 伸びしろ |
|---|---|---|
| **Wifi設備** | **2.90** | 2.10 |
| オンライン予約のしやすさ | 2.97 | 2.03 |
| ホテルの立地 | 2.98 | 2.02 |
| 食事の満足度 | 3.19 | 1.81 |
| 清潔さ | 3.23 | 1.77 |
| 送迎の便利さ | 3.24 | 1.76 |
| チェックイン・アウトの接客 | 3.25 | 1.75 |
| 滞在時の快適さ | 3.48 | 1.52 |
| その他のサービス | 3.65 | 1.35 |

### 不満の集中先は「個人客」

| 区分 | 件数 | 満足度 | 高評価率 | 低評価率 |
|---|---|---|---|---|
| グループ | 330 | 3.41 | 48.5% | 16.1% |
| **個人** | **170** | **2.91** | 30.0% | **33.5%** |

データ中で最も差が大きいセグメント。個人客は全体の34%を占めながら、低評価率が2倍以上ある。

さらに、個人客は**満足度を決める項目2つで劣っている**。

| 項目 | グループ | 個人 |
|---|---|---|
| Wifi設備 | 3.05 | **2.60** |
| 滞在時の快適さ | 3.67 | **3.12** |
| 清潔さ | 3.35 | 2.99 |

### その他のセグメント

| 宿泊カテゴリ | 件数 | 満足度 | 高評価率 |
|---|---|---|---|
| ビジネス | 102 | **3.44** | 52.0% |
| 旅行 | 160 | 3.33 | 47.5% |
| 学生 | 117 | 3.09 | 34.2% |
| 航空関係 | 79 | 3.09 | 35.4% |
| その他 | 42 | 3.10 | 33.3% |

年代では50代（3.46）と40代（3.37）が高く、10代（2.98）と60代（3.05）が低い。

**性別の差はほぼない**（女性3.26・男性3.22）。差がなかったことも報告に値する。

### 低評価層と高評価層を分けている項目

| 項目 | 低評価層 | 高評価層 | 差 |
|---|---|---|---|
| 滞在時の快適さ | 1.70 | 4.51 | **2.81** |
| 清潔さ | 1.78 | 4.10 | **2.32** |
| 食事の満足度 | 2.00 | 4.01 | **2.01** |
| Wifi設備 | 2.09 | 3.64 | 1.55 |
| オンライン予約のしやすさ | 2.63 | 3.38 | 0.74 |
| その他のサービス | 3.28 | 3.87 | 0.59 |
| チェックイン・アウトの接客 | 3.08 | 3.41 | 0.33 |
| ホテルの立地 | 2.95 | 3.28 | 0.32 |
| 送迎の便利さ | 3.28 | 3.50 | 0.23 |

送迎・立地・接客は、満足した客と不満な客でほとんど差がない。**これらは満足度の分かれ目になっていない。**

## 第3部　重回帰分析

### 変数設計

| | 内容 |
|---|---|
| 目的変数 | 満足度（1〜5、変換なし） |
| 説明変数 | 評価9項目 |
| 対象行 | 471行（異常値を含む29件を除外） |

満足度は1〜5の狭い範囲で歪みも小さいため、対数変換は不要。

### 結果

決定係数 **R² = 0.9217**（補正R² = 0.9202）

| 項目 | 回帰係数 | t値 | 有意 | VIF |
|---|---|---|---|---|
| 滞在時の快適さ | **0.3911** | 27.19 | ○ | 2.22 |
| Wifi設備 | **0.3643** | 27.49 | ○ | 1.77 |
| その他のサービス | 0.1363 | 11.59 | ○ | 1.09 |
| 清潔さ | 0.1359 | 8.84 | ○ | 2.47 |
| 食事の満足度 | 0.0864 | 6.44 | ○ | 1.93 |
| チェックイン・アウトの接客 | −0.0408 | −3.82 | ○ | 1.18 |
| オンライン予約のしやすさ | −0.0234 | −1.69 | | 2.06 |
| 送迎の便利さ | −0.0100 | −0.85 | | 1.61 |
| ホテルの立地 | 0.0034 | 0.27 | | 1.56 |

**9項目のうち、満足度を動かしているのは実質5項目。** 送迎・オンライン予約・立地は統計的に有意ではない。

VIFはいずれも2.5以下で、多重共線性は問題にならない水準。

### この演習の核心：単相関では順位が変わる

満足度との相関だけを見て改善項目を選ぶと、投資先を間違える。

| 項目 | 単相関 | 相関順位 | 回帰係数 | 係数順位 | 順位の動き |
|---|---|---|---|---|---|
| 滞在時の快適さ | 0.808 | 1 | 0.3911 | 1 | — |
| **清潔さ** | 0.695 | **2** | 0.1359 | **4** | ↓2 |
| 食事の満足度 | 0.605 | 3 | 0.0864 | 5 | ↓2 |
| **Wifi設備** | 0.585 | **4** | 0.3643 | **2** | ↑2 |
| オンライン予約のしやすさ | 0.268 | 5 | −0.0234 | 8 | ↓3 |
| **その他のサービス** | 0.219 | **6** | 0.1363 | **3** | ↑3 |
| ホテルの立地 | 0.157 | 7 | 0.0034 | 6 | ↑1 |
| チェックイン・アウトの接客 | 0.098 | 8 | −0.0408 | 9 | ↓1 |
| 送迎の便利さ | 0.087 | 9 | −0.0100 | 7 | ↑2 |

**清潔さは相関2位だが係数4位。** 理由は項目間の相関にある。

| 組み合わせ | 相関 |
|---|---|
| 清潔さ × 滞在時の快適さ | 0.703 |
| 食事の満足度 × 清潔さ | 0.656 |
| Wifi設備 × オンライン予約のしやすさ | 0.631 |

清潔さは「滞在時の快適さ」と強く相関している。清潔さが高い客は快適さも高く付けるため、単相関では快適さの効果を借りて強く見える。両方を同時に投入すると、清潔さ固有の効果は0.136に落ちる。

![png](img/10/図2_単相関が強く見える仕組み.png)

　清潔さの単相関が強く見える仕組み。実線（オレンジ）が迂回路、点線が清潔さ固有の効果。単相関はこの2つを合算して見せる。

逆に**その他のサービスは相関6位だが係数3位**。他項目との相関が低い（VIF 1.09）ため単相関では埋もれるが、独立した効果を持っている。

### 符号が説明できない項目

チェックイン・アウトの接客は係数 **−0.0408（t = −3.82）** で統計的に有意だが負。

単相関は +0.098 で正なので、符号が逆転している。これは「接客を悪くすれば満足度が上がる」という意味ではない。

- 効果量が極小（他項目の10分の1）
- 説明のつく仮説がない

**統計的に有意であることと、施策の根拠になることは別。** この項目は提案に使わない、という判断が正しい。

### 異常値を残した場合との比較

| モデル | n | R² | 結果 |
|---|---|---|---|
| そのまま投入（0と11.54を残す） | 500 | 0.9203 | 係数の順位は変わらない |
| 異常値を除外 | 471 | 0.9217 | こちらを採用 |

**今回は結論が変わらなかった。** しかし確かめたこと自体が前処理の仕事であり、確かめずに同じ結論に至った答案と、確かめて同じ結論に至った答案は評価が異なる。

## 第4部　優先度と施策効果

### 優先度マトリクス：影響度 × 伸びしろ

| 項目 | 影響度（係数） | 現在平均 | 伸びしろ | 優先度スコア | 判定 |
|---|---|---|---|---|---|
| **Wifi設備** | 0.3643 | 2.90 | 2.10 | **0.765** | 最優先 |
| 滞在時の快適さ | 0.3911 | 3.50 | 1.50 | 0.587 | 最優先 |
| 清潔さ | 0.1359 | 3.26 | 1.74 | 0.236 | 次善 |
| その他のサービス | 0.1363 | 3.65 | 1.35 | 0.184 | 次善 |
| 食事の満足度 | 0.0864 | 3.21 | 1.79 | 0.155 | 次善 |
| ホテルの立地 | 0.0034 | 2.97 | 2.03 | 0.007 | 投資不要 |
| 送迎の便利さ | −0.0100 | 3.25 | 1.75 | −0.018 | 投資不要 |
| オンライン予約のしやすさ | −0.0234 | 2.95 | 2.05 | −0.048 | 投資不要 |
| チェックイン・アウトの接客 | −0.0408 | 3.23 | 1.77 | −0.072 | 投資不要 |

**Wifiが1位になる理由**：影響度は2位（0.364）だが、現在平均が9項目中**最下位**（2.90）。影響の大きい項目で、最も点数が低い。

滞在時の快適さは影響度1位だが、すでに3.50まで来ているため伸びしろが小さい。

### 施策シミュレーション

回帰式に改善幅を入れ、予測満足度を四捨五入して高評価率・低評価率を再計算した。

| 施策 | 高評価率 | 低評価率 | 高評価率の変化 |
|---|---|---|---|
| 現状（施策なし） | 42.7% | 21.0% | — |
| Wifi設備 +0.5 | 48.8% | 18.0% | +6.1pt |
| **Wifi設備 +1.0** | **55.4%** | **14.9%** | **+12.7pt** |
| 滞在時の快適さ +0.5 | 47.6% | 17.8% | +4.9pt |
| Wifi +0.5 かつ 快適さ +0.5 | 57.1% | 14.6% | +14.4pt |
| 清潔さ +1.0 | 46.9% | 19.1% | +4.2pt |
| 送迎の便利さ +1.0 | 42.5% | 21.4% | −0.2pt |
| ホテルの立地 +1.0 | 42.7% | 21.0% | ±0pt |

**送迎と立地を1点上げても高評価率は動かない。** ここへの投資は満足度に返ってこない。

### 個人客に絞った効果

| 条件 | 個人客の高評価率 |
|---|---|
| 現状 | 32.3% |
| 個人客のWifiをグループ客水準（+0.45）に | 36.1% |

## 結論

### 現状 — 「平均3.24の裏で、2割が低評価を付けている」

- 満足度の平均3.24、中央値3.0。高評価42.2%、低評価22.0%
- 低評価層110件が点数を押し下げている。この層を減らすのが平均を上げる最短経路
- 個人客の低評価率は33.5%、グループ客は16.1%。個人客は170件（全体の34%）
- 評価項目で最も低いのはWifi設備（2.90）

平均を0.1上げる議論ではなく、低評価の2割が何に不満なのかを見る。**不満の集中先は「個人客」と「Wifi」である。**

### 分析 — 「相関の順位で選ぶと、投資先を間違える」

- 重回帰（n=471）の係数：滞在時の快適さ0.391、Wifi設備0.364、その他のサービス0.136、清潔さ0.136、食事0.086
- 送迎（−0.010）・オンライン予約（−0.023）・立地（0.003）はいずれも有意でない
- 清潔さは相関2位（0.695）だが係数4位（0.136）。快適さとの相関0.703による見かけの強さ
- その他のサービスは相関6位（0.219）だが係数3位（0.136）。単相関では埋もれる
- 低評価層と高評価層の差：快適さ2.81点・清潔さ2.32・食事2.01に対し、送迎0.23・立地0.32・接客0.33
- シミュレーション：Wifiを1点上げると高評価率42.7%→55.4%。送迎を1点上げても42.5%で不変

**9項目のうち満足度を動かすのは実質5項目。残り4項目に投資しても点数は返ってこない。そして動く5項目のうち、最も点数が低いのがWifiである。**

### 提案 — レビュー改善2つ・集客改善1つ

#### 提案1（最優先）　Wifi設備への投資を最優先にする

| | |
|---|---|
| 根拠 | 影響度は9項目中2位（係数0.364）、現在平均は9項目中最下位（2.90）。影響度×伸びしろのスコアで1位 |
| 期待効果 | 1点改善で高評価率 **+12.7pt**（42.7%→55.4%）、低評価率 **−6.1pt**（21.0%→14.9%） |
| 障壁 | 客室単位の増強は設備投資になる。低評価が集中する個人客の滞在エリアから部分的に実施し、再調査で効果を確認してから全館に広げる |

#### 提案2　個人客の体験を立て直す

| | |
|---|---|
| 根拠 | 個人客170件の満足度2.91・低評価率33.5%（グループ客は3.41・16.1%）。個人客はWifi 2.60（グループ3.05）、快適さ3.12（同3.67）と、効く項目2つで劣る |
| 期待効果 | 個人客のWifiをグループ客水準に揃えると、個人客の高評価率は32.3%→36.1%。低評価の3分の1がこの層にあるため、レビュー点数への寄与は大きい |
| 障壁 | 同じ設備で評価が分かれる理由がデータから分からない。客室の割り当て方か、繁忙帯の混雑か、現場へのヒアリングが先に必要 |

#### 提案3　集客はビジネス・40〜50代・グループに絞る

| | |
|---|---|
| 根拠 | 満足度が高いのはビジネス3.44（高評価率52.0%）、50代3.46、40代3.37、グループ3.41。低いのは学生3.09・航空関係3.09・その他3.10、10代2.98、個人2.91 |
| 期待効果 | 満足している層を増やせば、追加投資なしでレビュー点数の構成が改善する |
| 障壁 | 学生・個人客を切り捨てる判断ではない。提案1・2で体験が直るまでは満足層への集客を優先するという、順序の話として扱う |

### 欄外：この分析の限界

1. **相関であって因果ではない。** 満足度調査は同じ回答者が同じ時点で付けた点数であり、「Wifiを改善すれば満足度が上がる」は証明されていない。施策後の再調査が必要
2. **チェックイン・アウトの接客は有意だが負の係数**（−0.041、t=−3.82）。説明がつかないため施策の根拠にしない。有意性と重要性は別物
3. **決定係数0.922は満足度調査としては高すぎる。** このデータは満足度が評価項目からほぼ算出される構造になっており、実際の調査データでここまで綺麗に要因が分離することはない
4. **70代5件・80代1件・年代不明7件**は母数が小さく、率での比較に使えない
5. **レビュー点数と予約数の関係は、このデータに入っていない。** 「点数が上がれば集客が増える」は仮定であり、検証していない

### 効果検証に必要な追加データ

- 施策の実施日（Wifi増強の工事日など。前後比較の基準点）
- 予約経路と予約時のレビュー点数（点数と集客の関係を測るため）
- 客室タイプと階数（同じ設備で個人客の評価が低い理由の切り分け）
- 宿泊日と繁忙度（混雑が体験を下げているかの確認）
- 自由記述のコメント（点数だけでは不満の中身が分からない）

### More

- **単相関の高い順に改善項目を選んでいる**（清潔さを最優先にしている）
- 立地の異常値に気づいていない（11.54を1〜5として平均に入れている）
- 送迎・立地の改善を提案に入れている（影響度がほぼ0の項目）
- 80代の満足度が最低と報告している（母数1件）

### Good

- **「影響度が低い項目には投資しない」という逆方向の提案**を含んでいる
- 有意だが符号が説明できない接客の係数を、あえて採用しなかったと述べている
- 0を含めた平均と除いた平均の両方を出して、差を確認している

###  自身の回答の確認質問

効く順。

1. 「なぜ清潔さではなくWifiを最優先にしたのですか」
2. 「その高評価率+12.7ptはどうやって出した数字ですか」
3. 「Wifiを直せば本当に満足度が上がると言えますか」
4. 「80代の満足度が最低と書いていますが、母数は何件ですか」
5. 「送迎の評価が3.24なのはなぜ500件の平均になっていないのですか」